# L8c: Mitogen-Activated Protein Kinase (MAPK) Pathway
In this lecture we'll consider on the most foundational signal transduction pathways in human cells, the mitogen activated protein kinase (MAPK) pathway. The key ideas for today are:

* __MAPK Pathway__: A core signaling pathway that transmits signals from the cell surface to the nucleus, 
where it can regulate gene expression. The MAPK pathway is involved in a wide range of cellular processes, including cell proliferation, differentiation, and survival. Dysregulation of the MAPK pathway has been implicated in a number of diseases, including cancer, neurodegenerative disorders, and inflammatory diseases. 
* __Structure__: In humans, the MAPK pathway consists of at least four main cascades: ERK, JNK, p38, and ERK5. Each cascade is composed of three tiers of kinases (MAP3K, MAPKK, and MAPK) that are sequentially activated in response to specific extracellular stimuli.
* __The ERK1/2 pathway__ is one best-characterized MAPK system in mammals, typically activated by growth factors and involved in cell proliferation and differentiation. In contrast, the JNK and p38 pathways are primarily responsive to stress stimuli and play roles in adaptation to stress and apoptosis. The ERK5 pathway is less well understood, but is thought to be involved in cell survival and differentiation.

However, before talk about the MAPK pathway, let's review our model of the PhoR/PhoB two-component system, and fix the issue we identified in the last lecture.

## Problem Set 3 (PS3): PhoR/PhoB Two-Component System
The PhoR/PhoB two-component system is a well-studied example of a TCS that regulates bacteria's response to phosphate limitation. PhoR is the sensor kinase that detects low phosphate levels, while PhoB is the response regulator that activates phosphate acquisition and metabolism genes. 

[PhoR/PhoB Cartoon, Figure 7.5 reproduced from the textbook, "Molecular Biology of the Cell" by Alberts et al. 6th Edition](https://github.com/varnerlab/CHEME-5450-Lectures-Spring-2025/blob/main/lectures/week-8/L8b/figs/figure%207-05.jpg).

Review of the PhoR/PhoB TCS in _Escherichia coli_:
* [Gardner SG, McCleary WR. Control of the phoBR Regulon in Escherichia coli. EcoSal Plus. 2019 Sep;8(2):10.1128/ecosalplus.ESP-0006-2019. doi: 10.1128/ecosalplus.ESP-0006-2019. PMID: 31520469; PMCID: PMC11573284.](https://pubmed.ncbi.nlm.nih.gov/31520469/)

### Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading any needed resources, such as sample datasets, and setting up any required constants. 
* The `Include.jl` file also loads external packages, various functions that we will use in the exercise, and custom types to model the components of our problem. It checks for a `Manifest.toml` file; if it finds one, packages are loaded. Other packages are downloaded and then loaded.

In [1]:
include("Include.jl");

__Build the model__. To store all the problem data, we created [the `MyPrimalFluxBalanceAnalysisCalculationModel` type](src/Types.jl). Let's build one of these objects for our problem and store it in the `model::MyPrimalFluxBalanceAnalysisCalculationModel` variable. We also return the `rd::Dict{String, String}` dictionary, which maps the reaction name field (key) to the reaction string (value).
* __Builder (or factory) pattern__: For all custom types that we make, we'll use something like [the builder software pattern](https://en.wikipedia.org/wiki/Builder_pattern) to construct and initialize these objects. The calling syntax will be the same for all types: [a `build(...)` method](src/Factory.jl) will take the kind of thing we want to build in the first argument, and the data needed to make that type as [a `NamedTuple` instance](https://docs.julialang.org/en/v1/base/base/#Core.NamedTuple) in the second argument.
* __What's the story with the `let` block__? A [let block](https://docs.julialang.org/en/v1/manual/variables-and-scoping/#Let-Blocks) creates a new hard scope and new variable bindings each time they run. Thus, they act like a private scratch space, where data comes in (is captured by the block), but only what we want to be exposed comes out. 

In [2]:
model, rd = let

    # first, load the reaction file - and process it
    listofreactions = read_reaction_file(joinpath(_PATH_TO_DATA, "PHOB-3-5-2011.net")); # load the reactions from the VFF reaction file
    S, species, reactions, rd = build_stoichiometric_matrix(listofreactions); # Builds the stochiometric matrix, species list, and the reactions list
    boundsarray = build_default_bounds_array(listofreactions); # Builds a default bounds model using the flat file flags

    # build the FBA model -
    model = build(MyPrimalFluxBalanceAnalysisCalculationModel, (
        S = S, # stoichiometric matrix
        fluxbounds = boundsarray, # these are the *default* bounds, we'll need to update with new info if we have it
        species = species, # list of species. The rows of S are in this order
        reactions = reactions, # list of reactions. The cols of S are in this order
        objective = length(reactions) |> R -> zeros(R), # this is empty, we'll need to set this
    ));

    # return -
    model, rd
end;

`Unhide` the code block below to see how we build a table of the reactions in the model [using the `pretty_tables(...)` method exported from the `PrettyTables.jl` package](https://github.com/ronisbr/PrettyTables.jl).

In [3]:
let
    df = DataFrame()
    reactions = model.reactions;

    for i ∈ eachindex(reactions)
        reactionstring = reactions[i] |> key -> rd[key];
        row_df = (
            name = reactions[i],
            string = reactionstring,
        );
        push!(df, row_df);
    end

    pretty_table(df, tf = tf_simple, alignment = :l)
end

=============================== ============================================================
  name                           string                                                    
  String                         String                                                    
=============================== ============================================================
  PORIN_SYNTHESIS                [] = PORIN
  PORIN_TRANSLOCATION_MEMBRANE   PORIN = PORIN_MEMBRANE
  PORIN_DEGRADATION              PORIN = []
  PORIN_MEMBRANE_DEGRADATION     PORIN_MEMBRANE = []
  PhoR_SYNTHESIS                 [] = PhoR
  PhoR_LOCALIZATION_MEMBRANE     PhoR = PhoR_MEMBRANE
  PhoR_DEGRADATION               PhoR = []
  PhoR_MEMBRANE_DEGRADATION      PhoR_MEMBRANE = []
  PhoB_SYNTHESIS                 [] = PhoB
  PhoB_DEGRADATION               PhoB = []
  PhoB_PPASE_SYNTHESIS           [] = PhoB_PPASE
  PhoB_PPASE_DEGRADATION         PhoB_PPASE = []
  RNAP_ASSEMBLY                  [] = RNAP
  RNAP_DEGRADA

Update the objective function. Select a process to maximize:

In [4]:
i = findfirst(x-> x=="PHOA_TRANSLATION", model.reactions);
objective = model.objective;
objective[i] = 1;

### The bounds trick
Translation occurs without transcription. Alternatively, transcription occurs without PhoB activation. How are we going to fix this? Answer: there is a trick with the bounds (that incorporates many things we have been exploring) that we can use to fix this problem:
* [Vilkhovoy M, Horvath N, Shih CH, Wayman JA, Calhoun K, Swartz J, Varner JD. Sequence-Specific Modeling of E. coli Cell-Free Protein Synthesis. ACS Synth Biol. 2018 Aug 17;7(8):1844-1857. doi: 10.1021/acssynbio.7b00465. Epub 2018 Jul 16. PMID: 29944340.](https://www.biorxiv.org/content/10.1101/139774v2)

Fill me in here with the logic of the bounds trick.

In [ ]:
rX,rL = let 
    
    # compute the kinetic limit of transcription and translation 
    

end

### Compute the optimal flux distribution 

Finally, let's compute the optimal metabolic distribution $\left\{\hat{v}_{i} \mid i = 1,2,\dots,\mathcal{R}\right\}$ by solving the [linear programming problem](). We solve the optimization problem by passing the `model::MyPrimalFluxBalanceAnalysisCalculationModel` to [the `solve(...)` method](src/Compute.jl). This method returns a `solution::Dict{String, Any}` dictionary, which holds information about the solution.
* __Why the [try-catch environment](https://docs.julialang.org/en/v1/base/base/#try)__? The [solve(...) method](src/Compute.jl) has an [@assert statement](https://docs.julialang.org/en/v1/base/base/#Base.@assert) to check if the calculation has converged. Thus, the solve method can [throw](https://docs.julialang.org/en/v1/base/base/#Core.throw) an [AssertionError](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) if the optimization problem fails to converge. To gracefully handle this case, we use a [try-catch construct](https://docs.julialang.org/en/v1/base/base/#try). See the [is_solved_and_feasible method from the JuMP package](https://jump.dev/JuMP.jl/stable/api/JuMP/#JuMP.is_solved_and_feasible) for more information.

In [5]:
solution = let
    
    solution = nothing; # initialize nothing for the solution
    try
        solution = solve(model); # call the solve method with our problem model -
    catch error
        println("error: $(error)"); # Oooooops! Looks like we have a *major malfunction*, problem didn't solve
    end

    # return solution
    solution
end;

__Flux table__: Let's use [the `pretty_tables(...)` method exported by the `PrettyTables.jl` package](https://github.com/ronisbr/PrettyTables.jl) to display the estimated optimal metabolic fluxes. `Unhide` the code block below to see how we constructed the flux table.

In [6]:
let

    # setup -
    S = model.S;
    flux_bounds_array = model.fluxbounds;
    number_of_reactions = size(S,2); # columns
	flux_table = Array{Any,2}(undef,number_of_reactions,5)
    flux = solution["argmax"];
    
    # populate the state table -
	for reaction_index = 1:number_of_reactions
		flux_table[reaction_index,1] = model.reactions[reaction_index]
		flux_table[reaction_index,2] = flux[reaction_index]
		flux_table[reaction_index,3] = flux_bounds_array[reaction_index,1]
		flux_table[reaction_index,4] = flux_bounds_array[reaction_index,2]
        flux_table[reaction_index,5] = model.reactions[reaction_index] |> key-> rd[key]
	end

    # header row -
	flux_table_header_row = (["Reaction","v̂ᵢ", "v̂ᵢ LB", "v̂ᵢ UB", "Reaction"],["","mmol/gDW-time", "mmol/gDW-time", "mmol/gDW-time", "N/A"]);
		
	# write the table -
	pretty_table(flux_table; header=flux_table_header_row, tf=tf_simple, alignment = :l)
end

=============================== =============== =============== =============== ============================================================
  Reaction                       v̂ᵢ              v̂ᵢ LB           v̂ᵢ UB           Reaction                                                  
                                 mmol/gDW-time   mmol/gDW-time   mmol/gDW-time   N/A                                                       
=============================== =============== =============== =============== ============================================================
  PORIN_SYNTHESIS                0.0             0.0             1000.0          [] = PORIN
  PORIN_TRANSLOCATION_MEMBRANE   0.0             0.0             1000.0          PORIN = PORIN_MEMBRANE
  PORIN_DEGRADATION              0.0             0.0             1000.0          PORIN = []
  PORIN_MEMBRANE_DEGRADATION     0.0             0.0             1000.0          PORIN_MEMBRANE = []
  PhoR_SYNTHESIS                 0.0          

## MAPK Pathway Overview
The MAPK pathway is a core signaling pathway that transmits signals from the cell surface to the nucleus, where it can regulate gene expression.

Let's take a look at a review of the structure of the MAPK pathway, and particularly the ERK1/2 pathway, which is failry well characterized in mammals.
* [Martin-Vega A, Cobb MH. Navigating the ERK1/2 MAPK Cascade. Biomolecules. 2023 Oct 20;13(10):1555. doi: 10.3390/biom13101555. PMID: 37892237; PMCID: PMC10605237.](https://pubmed.ncbi.nlm.nih.gov/37892237/)

## MAPK and the G1/S Transition in the Cell Cycle
The G1/S transition is a critical checkpoint in the cell cycle that determines whether a cell will proceed to DNA replication and division (irreversible commitment to cell division). 
* The G1/S transition is regulated by a number of signaling pathways, including the MAPK pathway. In response to mitogenic (growth) signals, the MAPK pathway is activated, leading to the expression of genes that promote cell cycle progression. Dysregulation of the MAPK pathway can lead to uncontrolled cell proliferation and cancer.
* Cyclin-dependent kinase 4 (CDK4) and CDK6 are critical mediators of cellular transition into S phase and are important for the initiation, growth and survival of many cancer types. Pharmacological inhibitors of CDK4/6 have rapidly become a new standard of care for patients with advanced hormone receptor-positive breast cancer. As expected, CDK4/6 inhibitors arrest sensitive tumour cells in the G1 phase of the cell cycle. However, the effects of CDK4/6 inhibition are far more wide-reaching. 

Let's look at the role of the MAPK pathway, and the activity of CDK4/6 in the G1/S transition in more detail.
* [Goel, S., Bergholz, J.S. & Zhao, J.J. Targeting CDK4 and CDK6 in cancer. Nat Rev Cancer 22, 356–372 (2022). https://doi.org/10.1038/s41568-022-00456-3](https://pubmed.ncbi.nlm.nih.gov/35304604/)

We (and many others) have built computational models of cell cycle progression, including the G1/S transition. These models can be used to study the effects of different perturbations on cell cycle progression, and to identify potential therapeutic targets for cancer treatment.
* [Nayak S, Salim S, Luan D, Zai M, Varner JD. A test of highly optimized tolerance reveals fragile cell-cycle mechanisms are molecular targets in clinical cancer trials. PLoS One. 2008 Apr 23;3(4):e2016. doi: 10.1371/journal.pone.0002016. PMID: 18431497; PMCID: PMC2291571.](https://pubmed.ncbi.nlm.nih.gov/18431497/)

## MAPK in Cell Differentiation Pathways
The MAPK pathway is also involved in cell differentiation, a process by which cells become specialized to perform specific functions. For example, consider the role of the MAPK pathway in the differentiation of HL-60 cells into neutrophils or macrophages.

* [Wang J, Yen A. A MAPK-positive feedback mechanism for BLR1 signaling propels retinoic acid-triggered differentiation and cell cycle arrest. J Biol Chem. 2008 Feb 15;283(7):4375-86. doi: 10.1074/jbc.M708471200. Epub 2007 Nov 15. PMID: 18006504.](https://pubmed.ncbi.nlm.nih.gov/18006504/)

We've developed several model of the MAPK pathway in cell differentiation, which can be used to study the effects of different perturbations on cell differentiation and to identify potential therapeutic targets.
* [Tasseff R, Nayak S, Song SO, Yen A, Varner JD. Modeling and analysis of retinoic acid induced differentiation of uncommitted precursor cells. Integr Biol. 2011 May;3(5):578-91. doi: 10.1039/c0ib00141d. Epub 2011 Mar 24. PMID: 21437295; PMCID: PMC3685823.](https://pubmed.ncbi.nlm.nih.gov/21437295/)
* [Tasseff R, Jensen HA, Congleton J, Dai D, Rogers KV, Sagar A, Bunaciu RP, Yen A, Varner JD. An Effective Model of the Retinoic Acid Induced HL-60 Differentiation Program. Sci Rep. 2017 Oct 30;7(1):14327. doi: 10.1038/s41598-017-14523-5. PMID: 29085021; PMCID: PMC5662654.](https://www.biorxiv.org/content/10.1101/138784v2)